In [1]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing import TypedDict, Annotated
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_10544\763961630.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
LLM = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)

In [3]:
#search-tool 
search_tool = DuckDuckGoSearchRun(
    name = "search-tool",
    description=(
        "if the search related to updated knowleged from internet"
        "use this tool when user asks about current events"
        "new, current information, information require"
        "use internet for search"
    )
)

In [4]:
@tool
def search_tool(query: str)->str:
    """ search the internet for the query required from the user """
    
    decision = interrupt({
        "type": "approval",
        "tool": "search-tool",
        "query": query,
        "message": "The AI wants to search the internet",
        "instruction": "Approve this question (y/n)" 
    })
    
    if decision['approval'] == 'n':
        return {"message": [AIMessage(content="sorry cant search this for you!!")]}
    
    return search_tool.invoke(query)

In [5]:
tool = [search_tool]

In [6]:
llm_with_tool = LLM.bind_tools(tool)

In [7]:
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [8]:
def chat_node(state: chatstate):
    """ Chatbot with a search tool """
    messages = state['messages']
    
    response = llm_with_tool.invoke(messages)
    
    return {"messages": [response]}

tools = ToolNode(tool)

In [9]:
graph = StateGraph(chatstate)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tools)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")

chatbot = graph.compile()


In [ ]:
while True:
    user_input = input("type_here...")
    
    print("you: ", user_input)
    
    if user_input.lower().strip() in ["exit", "break", "thanks you"]:
        break
    
    # implementing hilt
    